## Meta Ads

#### Useful links:
- [Get a Developer account](https://developers.facebook.com/async/registration/dialog/?src=default)
- [Identity and Location Confirmation](Facebook.com/ID)
- [Meta Ads API Documentation](https://www.facebook.com/ads/library/api/)
- [Meta Ads API - Ads Archive Reference](https://developers.facebook.com/docs/graph-api/reference/ads_archive/)


In [ ]:
# requirements
%pip install pandas==2.3.2 requests==2.32.5 python-dotenv==1.1.1 pandera==0.26.1

In [1]:
import os
import json
import glob
import pandas as pd
import pandera.pandas as pa
from pandera.errors import SchemaErrors
import requests
from dotenv import load_dotenv
from datetime import datetime
from time import sleep

In [2]:
import sys

print("Python version:")
print(sys.version)

Python version:
3.13.2 (main, Mar 31 2025, 14:41:24) [Clang 15.0.0 (clang-1500.3.9.4)]


In [ ]:
# environment variables
load_dotenv()
ACCESS_TOKEN = os.getenv("ACCESS_TOKEN")
FILEPATH = os.getenv("FILEPATH")
# Token expires in 05/03/2026


#### Meta Ads API Available Parameters


- ad_active_status: *enum {ACTIVE, ALL, INACTIVE}* <span style="color:red">**(DEFAULT: Active)**</span>
- ad_delivery_date_max: *str*
- ad_delivery_date_min: *str*
- ad_reached_countries: *array<enum> (countries abbreviations)*
- ad_type: *enum {ALL, EMPLOYMENT_ADS, FINANCIAL_PRODUCTS_AND_SERVICES_ADS, HOUSING_ADS, POLITICAL_AND_ISSUE_ADS}* <span style="color:red">**(DEFAULT: ALL)**</span>
- bylines: *array<str>*
- delivery_by_region: *array<str>*
- estimated_audience_size_max: *int64*
- estimated_audience_size_min: *int64*
- languages: *array<str>*
- media_type: *enum {ALL, IMAGE, MEME, VIDEO, NONE}*
- publisher_platforms: *array<enum {FACEBOOK, INSTAGRAM, AUDIENCE_NETWORK, MESSENGER, WHATSAPP, OCULUS, THREADS}>*
- search_page_ids: *array<int64>*
- search_terms: *str* <span style="color:red">**(DEFAULT: "")**</span>
- search_type: *enum {KEYWORD_UNORDERED, KEYWORD_EXACT_PHRASE}* <span style="color:red">**(DEFAULT: KEYWORD_UNORDERED)**</span>
- unmask_removed_content: *bool* <span style="color:red">**(DEFAULT: False)**</span>

In [ ]:
# Expected Payload
# It consists of a dict containing the access token and the desired parameters for the request
# Must have search_terms or search_page_ids parameters
# Please update the variable below with the desired search parameters
payload = {
            "access_token": ACCESS_TOKEN,
        }

#### Meta Ads API Available Fields

- All countries:
    * id
    * ad_creation_time
    * ad_creative_bodies
    * ad_creative_link_captions
    * ad_creative_link_descriptions
    * ad_creative_link_titles
    * ad_delivery_start_time
    * ad_delivery_stop_time
    * ad_snapshot_url
    * bylines
    * currency
    * delivery_by_region
    * demographic_distribution
    * estimated_audience_size
    * impressions
    * languages
    * page_id
    * page_name
    * publisher_platforms
    * spend
 
- Brazil only:
    * age_country_gender_reach_breakdown
    * br_total_reach
    * target_ages
    * target_gender
    * target_locations
    * total_reach_by_location
    
- EU only:
    * age_country_gender_reach_breakdown
    * beneficiary_payers
    * eu_total_reach
    * target_ages
    * target_gender
    * target_locations
    * total_reach_by_location


#### API Connection

In [ ]:
# fields should be a string of selected fields separeted by comma, without any blank space
# e.g: "id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions"
fields = ""

In [ ]:
# newest api version is v23.0 (08-09-2025)
api_url = f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"

In [4]:
# API Requests
def request_to_api(session, payload, api_url):
    response = session.get(api_url, params=payload)
    return response.json()

# Pagination
def paginate(session, payload, response):
    collected_ads = []
    while "paging" in response:
        next_page_url = response["paging"].get("next", "")
        sleep(1)
        response = request_to_api(session,payload, next_page_url)
        # The implementation of pagination ends here
        # Feel free to change implementation to extract data
        data = response.get("data", [])
        collected_ads.extend(data)
    return collected_ads

# Session
session = requests.Session()

### Special Criteria

**SC1: Does the platform provide an API to access its ad repository and extract data on advertising content?**

*This item verifies whether the platform provides an ad repository API with at least one endpoint for programmatically extracting advertising data. Full availability is confirmed when the API returns information on ads across all categories. The assessment should confirm that the endpoint allows the retrieval and storage of ad data without requiring privileged or internal access beyond standard developer registration.*


Yes, but only for political ads

In [5]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "saúde"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["BR"],
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": False
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response["data"])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df.head())
else:
    print("No data in response")
    print(response)

Got 25 ads in a single request


,id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,...,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location,bylines
0,679195485160563,2026-01-02,[This ad ran without a required disclaimer.],[This ad ran without a required disclaimer.],[This ad ran without a required disclaimer.],[This ad ran without a required disclaimer.],2026-01-02,2026-01-03,"[{'country': 'BR', 'age_gender_breakdowns': [{...",1657.0,...,[pt],134017799788024,Professor Pinheiro Neto,"[facebook, instagram, audience_network, messen...","{'lower_bound': '0', 'upper_bound': '99'}","[18, 65]",All,"[{'name': 'Ji-Paraná, RO, Brasil', 'num_obfusc...","[{'key': 'BR', 'value': 1657}]",NaN
1,1493697551691716,2026-01-02,[This ad ran without a required disclaimer.],[This ad ran without a required disclaimer.],[This ad ran without a required disclaimer.],[This ad ran without a required disclaimer.],2026-01-02,2026-01-03,"[{'country': 'BR', 'age_gender_breakdowns': [{...",4300.0,...,[pt],134017799788024,Professor Pinheiro Neto,"[facebook, instagram, audience_network, messen...","{'lower_bound': '0', 'upper_bound': '99'}","[18, 65]",All,"[{'name': 'Ji-Paraná, RO, Brasil', 'num_obfusc...","[{'key': 'BR', 'value': 4300}]",NaN
2,1382880096340198,2026-01-06,[📍 Um ano de mandato. Muitas histórias. Muita ...,NaN,NaN,NaN,2026-01-06,2026-01-07,"[{'country': 'BR', 'age_gender_breakdowns': [{...",4471.0,...,[pt],1817070968511062,Dudu Cunha,"[facebook, instagram]","{'lower_bound': '0', 'upper_bound': '99'}","[18, 65]",All,"[{'name': 'Indaial, SC, Brasil', 'num_obfuscat...","[{'key': 'BR', 'value': 4471}]",Dudu Cunha
3,1602049390971904,2026-01-07,[🧹 Uma faxineira… a identidade secreta de uma ...,[play.google.com],NaN,[Assista TODOS os episodios AGORA ❗👉👉],2026-01-07,NaN,"[{'country': 'PT', 'age_gender_breakdowns': [{...",NaN,...,[pt],628548157018486,Radreel-Funny&Touching,"[facebook, instagram, audience_network, messen...",NaN,"[18, 65]",All,"[{'name': 'Portugal', 'num_obfuscated': 0, 'ty...","[{'key': 'EU', 'value': 4}]",NaN
4,810935538671431,2026-01-07,[🧹 Uma faxineira… a identidade secreta de uma ...,[play.google.com],NaN,[Assista TODOS os episodios AGORA ❗👉👉],2026-01-07,NaN,"[{'country': 'PT', 'age_gender_breakdowns': [{...",NaN,...,[pt],628548157018486,Radreel-Funny&Touching,"[facebook, instagram, audience_network, messen...",NaN,"[18, 65]",All,"[{'name': 'Portugal', 'num_obfuscated': 0, 'ty...","[{'key': 'EU', 'value': 1}]",NaN


In [6]:
# Checking type of ad in response
political_fields = ["bylines", "currency", "delivery_by_region", "demographic_distribution", "estimated_audience_size", "impressions", "spend"]
political_ads = 0
other_ads = 0
for ad in response["data"]:
    has_political_fields = False
    for k in ad.keys():
        if k in political_fields:
            has_political_fields = True
            break
    if has_political_fields:
        political_ads += 1
    else:
        other_ads += 1

print("Total political ads:", political_ads)
print("Total other categories ads:", other_ads)

Total political ads: 4
Total other categories ads: 21


In [7]:
# EMPLOYMENT_ADS
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,page_id,page_name,publisher_platforms"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "saúde"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["BR"],
            "ad_type": "EMPLOYMENT_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response["data"])} in EMPLOYMENT_ADS in a single request")
    df = pd.DataFrame(response["data"])
    display(df.head())
else:
    print("No data found in EMPLOYMENT_ADS")
    print(response)

No data found in EMPLOYMENT_ADS
{'error': {'message': 'An unknown error has occurred.', 'type': 'OAuthException', 'code': 1, 'fbtrace_id': 'AzLDJ5www5IG61umOnDlrbW'}}


In [8]:
# FINANCIAL_PRODUCTS_AND_SERVICES_ADS
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,page_id,page_name,publisher_platforms"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "saúde"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["BR"],
            "ad_type": "FINANCIAL_PRODUCTS_AND_SERVICES_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response["data"])} in FINANCIAL_PRODUCTS_AND_SERVICES_ADS in a single request")
    df = pd.DataFrame(response["data"])
    display(df.head())
else:
    print("No data found in FINANCIAL_PRODUCTS_AND_SERVICES_ADS")
    print(response)

No data found in FINANCIAL_PRODUCTS_AND_SERVICES_ADS
{'error': {'message': 'An unknown error has occurred.', 'type': 'OAuthException', 'code': 1, 'fbtrace_id': 'Av4KAs7wfKfZY3sIr8OkEW2'}}


In [9]:
# HOUSING_ADS
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,page_id,page_name,publisher_platforms"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "saúde"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["BR"],
            "ad_type": "HOUSING_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response["data"])} in HOUSING_ADS in a single request")
    df = pd.DataFrame(response["data"])
    display(df.head())
else:
    print("No data found in HOUSING_ADS")
    print(response)

No data found in HOUSING_ADS
{'error': {'message': 'An unknown error has occurred.', 'type': 'OAuthException', 'code': 1, 'fbtrace_id': 'AUx2xeUnyOVuW_AI4bWuwAY'}}


In [10]:
# POLITICAL_AND_ISSUE_ADS
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "saúde"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["BR"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response["data"])} ads in POLITICAL_AND_ISSUE_ADS in a single request")
    df = pd.DataFrame(response["data"])
    display(df.head())
else:
    print("No data found in POLITICAL_AND_ISSUE_ADS")
    print(response)

Got 25 ads in POLITICAL_AND_ISSUE_ADS in a single request


,id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,currency,...,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location,bylines,ad_creative_link_descriptions
0,679195485160563,2026-01-02,[📢 CONCURSO DA PREFEITURA DE JI-PARANÁ CONFIRM...,[WhatsApp],[api.whatsapp.com],2026-01-02,2026-01-03,"[{'country': 'BR', 'age_gender_breakdowns': [{...",1657,BRL,...,134017799788024,Professor Pinheiro Neto,"[facebook, instagram, audience_network, messen...","{'lower_bound': '0', 'upper_bound': '99'}","[18, 65]",All,"[{'name': 'Ji-Paraná, RO, Brasil', 'num_obfusc...","[{'key': 'BR', 'value': 1657}]",NaN,NaN
1,1493697551691716,2026-01-02,[📢 CONCURSO DA PREFEITURA DE JI-PARANÁ CONFIRM...,[WhatsApp],[api.whatsapp.com],2026-01-02,2026-01-03,"[{'country': 'BR', 'age_gender_breakdowns': [{...",4300,BRL,...,134017799788024,Professor Pinheiro Neto,"[facebook, instagram, audience_network, messen...","{'lower_bound': '0', 'upper_bound': '99'}","[18, 65]",All,"[{'name': 'Ji-Paraná, RO, Brasil', 'num_obfusc...","[{'key': 'BR', 'value': 4300}]",NaN,NaN
2,1382880096340198,2026-01-06,[📍 Um ano de mandato. Muitas histórias. Muita ...,NaN,NaN,2026-01-06,2026-01-07,"[{'country': 'BR', 'age_gender_breakdowns': [{...",4471,BRL,...,1817070968511062,Dudu Cunha,"[facebook, instagram]","{'lower_bound': '0', 'upper_bound': '99'}","[18, 65]",All,"[{'name': 'Indaial, SC, Brasil', 'num_obfuscat...","[{'key': 'BR', 'value': 4471}]",Dudu Cunha,NaN
3,4198014317154825,2026-01-07,[Hoje damos um passo muito importante para Bom...,[www.instagram.com],NaN,2026-01-07,NaN,"[{'country': 'BR', 'age_gender_breakdowns': [{...",1406,BRL,...,355964377812210,Eures Ribeiro,"[facebook, instagram]","{'lower_bound': '0', 'upper_bound': '99'}","[25, 60]",All,"[{'name': 'Bom Jesus da Lapa, BA, Brasil', 'nu...","[{'key': 'BR', 'value': 1406}]",Eures Ribeiro,NaN
4,3915139728786569,2026-01-07,[RETROSPECTIVA 2025 | Saúde de Maracaju\nJanei...,NaN,NaN,2026-01-07,NaN,"[{'country': 'BR', 'age_gender_breakdowns': [{...",404,BRL,...,261610264541037,Marcos Calderan,"[facebook, instagram]","{'lower_bound': '0', 'upper_bound': '99'}","[20, 65]",All,"[{'name': 'Maracaju (disambiguation), MS, Bras...","[{'key': 'BR', 'value': 404}]",Marcos Calderan,NaN


**SC3: Can data from both active and inactive ads be extracted?**

*This item verifies whether the platform allows the extraction of ad data through either the GUI or the API, from at least one day after publication to at least one year prior. Full availability is confirmed when both active and inactive ad data are delivered across all ad categories. The assessment should test the interface and endpoints to confirm whether both active and inactive ads can be retrieved.*


Yes, both active and inactive

In [11]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "saúde"
active_payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ACTIVE",
            "ad_reached_countries": ["BR"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
inactive_payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "INACTIVE",
            "ad_reached_countries": ["BR"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
active_response = request_to_api(session, active_payload, api_url)
inactive_response = request_to_api(session, inactive_payload, api_url)
if "data" in active_response:
    print(f"Got {len(active_response["data"])} ACTIVE ads in a single request")
    active_df = pd.DataFrame(active_response["data"])
    display(active_df.head())
else:
    print("No ACTIVE ads were found")
    print(active_response)
if "data" in inactive_response:
    print(f"Got {len(inactive_response["data"])} INACTIVE ads in a single request")
    inactive_df = pd.DataFrame(inactive_response["data"])
    display(inactive_df.head())
else:
    print("No INACTIVE ads were found")
    print(inactive_response)


Got 25 ACTIVE ads in a single request


,id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_delivery_start_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,...,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location,ad_creative_link_titles,ad_creative_link_descriptions
0,4198014317154825,2026-01-07,[Hoje damos um passo muito importante para Bom...,[www.instagram.com],2026-01-07,"[{'country': 'BR', 'age_gender_breakdowns': [{...",1406,Eures Ribeiro,BRL,"[{'percentage': '1', 'region': 'Bahia'}]",...,355964377812210,Eures Ribeiro,"[facebook, instagram]","{'lower_bound': '0', 'upper_bound': '99'}","[25, 60]",All,"[{'name': 'Bom Jesus da Lapa, BA, Brasil', 'nu...","[{'key': 'BR', 'value': 1406}]",NaN,NaN
1,3915139728786569,2026-01-07,[RETROSPECTIVA 2025 | Saúde de Maracaju\nJanei...,NaN,2026-01-07,"[{'country': 'BR', 'age_gender_breakdowns': [{...",404,Marcos Calderan,BRL,"[{'percentage': '1', 'region': 'Mato Grosso do...",...,261610264541037,Marcos Calderan,"[facebook, instagram]","{'lower_bound': '0', 'upper_bound': '99'}","[20, 65]",All,"[{'name': 'Maracaju (disambiguation), MS, Bras...","[{'key': 'BR', 'value': 404}]",NaN,NaN
2,880311721154119,2025-12-05,[Atendimento gratuito!],NaN,2026-01-07,"[{'country': 'BR', 'age_gender_breakdowns': [{...",643,Dr Francisco Costa,BRL,"[{'percentage': '1', 'region': 'Piauí'}]",...,1695544247372940,Dr. Francisco,"[facebook, instagram]","{'lower_bound': '0', 'upper_bound': '99'}","[37, 65]",Women,"[{'name': 'Santana do Piauí, PI, Brasil', 'num...","[{'key': 'BR', 'value': 643}]",NaN,NaN
3,810319402043702,2026-01-07,"[Começamos o ano acelerando, dia histórico par...",[fb.com],2026-01-07,"[{'country': 'BR', 'age_gender_breakdowns': [{...",1450,Jeferson Timoteo,BRL,"[{'percentage': '0.561798', 'region': 'Paraíba...",...,103519928993290,Jeferson Timoteo,"[facebook, instagram]","{'lower_bound': '0', 'upper_bound': '99'}","[18, 65]",All,"[{'name': 'Camutanga, Brazil, PE, Brasil', 'nu...","[{'key': 'BR', 'value': 1450}]",NaN,NaN
4,1406129004558696,2026-01-06,"[Destinamos mais DE 2,3 MILHÕES em emendas par...",NaN,2026-01-07,"[{'country': 'BR', 'age_gender_breakdowns': [{...",1639,Helder Salomão,BRL,"[{'percentage': '1', 'region': 'Espírito Santo'}]",...,546851998788534,Helder Salomão,"[facebook, instagram]","{'lower_bound': '0', 'upper_bound': '99'}","[18, 65]",All,"[{'name': 'Mantenópolis, ES, Brasil', 'num_obf...","[{'key': 'BR', 'value': 1639}]",NaN,NaN


Got 25 INACTIVE ads in a single request


,id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,currency,...,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location,bylines,ad_creative_link_descriptions
0,679195485160563,2026-01-02,[📢 CONCURSO DA PREFEITURA DE JI-PARANÁ CONFIRM...,[WhatsApp],[api.whatsapp.com],2026-01-02,2026-01-03,"[{'country': 'BR', 'age_gender_breakdowns': [{...",1657,BRL,...,134017799788024,Professor Pinheiro Neto,"[facebook, instagram, audience_network, messen...","{'lower_bound': '0', 'upper_bound': '99'}","[18, 65]",All,"[{'name': 'Ji-Paraná, RO, Brasil', 'num_obfusc...","[{'key': 'BR', 'value': 1657}]",NaN,NaN
1,1493697551691716,2026-01-02,[📢 CONCURSO DA PREFEITURA DE JI-PARANÁ CONFIRM...,[WhatsApp],[api.whatsapp.com],2026-01-02,2026-01-03,"[{'country': 'BR', 'age_gender_breakdowns': [{...",4300,BRL,...,134017799788024,Professor Pinheiro Neto,"[facebook, instagram, audience_network, messen...","{'lower_bound': '0', 'upper_bound': '99'}","[18, 65]",All,"[{'name': 'Ji-Paraná, RO, Brasil', 'num_obfusc...","[{'key': 'BR', 'value': 4300}]",NaN,NaN
2,1382880096340198,2026-01-06,[📍 Um ano de mandato. Muitas histórias. Muita ...,NaN,NaN,2026-01-06,2026-01-07,"[{'country': 'BR', 'age_gender_breakdowns': [{...",4471,BRL,...,1817070968511062,Dudu Cunha,"[facebook, instagram]","{'lower_bound': '0', 'upper_bound': '99'}","[18, 65]",All,"[{'name': 'Indaial, SC, Brasil', 'num_obfuscat...","[{'key': 'BR', 'value': 4471}]",Dudu Cunha,NaN
3,2070918863655713,2026-01-07,[Um dos temas que eu mais falei com você aqui ...,NaN,NaN,2026-01-07,2026-01-07,"[{'country': 'BR', 'age_gender_breakdowns': [{...",1,BRL,...,323918771004168,Paulo Serra,[instagram],"{'lower_bound': '0', 'upper_bound': '99'}","[18, 65]",All,"[{'name': 'Campinas, SP, Brasil', 'num_obfusca...","[{'key': 'BR', 'value': 1}]",Paulo Serra,NaN
4,1108132294644554,2026-01-06,[Sua empresa só funciona quando você está pres...,[danielreiche.com.br],[5 sinais de que sua empresa precisa de estrat...,2026-01-06,2026-01-07,"[{'country': 'BR', 'age_gender_breakdowns': [{...",160,BRL,...,114074420304523,danielgomesreiche,[instagram],"{'lower_bound': '0', 'upper_bound': '99'}","[24, 58]",All,"[{'name': 'Brasil', 'num_obfuscated': 0, 'type...","[{'key': 'BR', 'value': 160}]",NaN,[Deixe seu negócio crescer sem você]


### ACCESSIBILITY


**OC3: Can the requested data be extracted directly from the ad repository response?**

This item verifies whether the platform ad repository returns structured data on ad creatives and authorship directly in the response, rather than providing a link that redirects to the data. Audiovisual media files and data (e.g., images, videos, and audio) should not be considered when assessing this item. The assessment should examine sample data responses from both the ad repository GUI and API to confirm that the requested public data is included in the returned payload.*


Yes.

In [12]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,ad_creative_bodies,ad_creative_link_descriptions,ad_creative_link_titles,bylines,page_id,page_name"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "saúde"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["BR"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response["data"])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df.head())
else:
    print("No ads were found")
    print(response)

Got 25 ads in a single request


,id,ad_creative_bodies,ad_creative_link_titles,page_id,page_name,bylines,ad_creative_link_descriptions
0,679195485160563,[📢 CONCURSO DA PREFEITURA DE JI-PARANÁ CONFIRM...,[api.whatsapp.com],134017799788024,Professor Pinheiro Neto,NaN,NaN
1,1493697551691716,[📢 CONCURSO DA PREFEITURA DE JI-PARANÁ CONFIRM...,[api.whatsapp.com],134017799788024,Professor Pinheiro Neto,NaN,NaN
2,1382880096340198,[📍 Um ano de mandato. Muitas histórias. Muita ...,NaN,1817070968511062,Dudu Cunha,Dudu Cunha,NaN
3,4198014317154825,[Hoje damos um passo muito importante para Bom...,NaN,355964377812210,Eures Ribeiro,Eures Ribeiro,NaN
4,3915139728786569,[RETROSPECTIVA 2025 | Saúde de Maracaju\nJanei...,NaN,261610264541037,Marcos Calderan,Marcos Calderan,NaN


In [13]:
if response["data"]:
    URL_PATTERN = r"https?://\S+"
    df = pd.DataFrame(response["data"])
    schema = pa.DataFrameSchema(
        columns={},
        checks=pa.Check(
            lambda df: (
                df.select_dtypes(include=['object'])
                .apply(lambda s: s.str.contains(URL_PATTERN).any(), axis=0)
                .sum() == 0
            ),
            name="DirectResponse"
        ),
        strict=False 
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: No links found in any ad creative or authorship column.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Links are present in ad creatives or authorship columns.")
        print(err)

✅ Validation Successful: No links found in any ad creative or authorship column.


**OC5: Can data from an individual ad be retrieved from the platform?**

*This item verifies whether it is possible to retrieve data from a specific advertisement on the ad repository using a unique identifier, rather than relying on search terms or other parameters and filters. The assessment should review the ad repository documentation and test available features to confirm that an individual ad can be retrieved directly by its unique identifier.*



Ad Library link: [AD 1362986028521549](https://www.facebook.com/ads/library/?id=1362986028521549)

No. There's no option to retrieve an ad by id

In [14]:
# Use this cell to develop an example where a request for ads from a campaign is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
ad_id = "1362986028521549"
fields="id"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["BR"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_terms": ad_id,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    found_ad = False
    for ad in response["data"]:
        if ad["id"] == ad_id:
            found_ad = True
            break
    if found_ad:
        print(f"Found ad {ad_id} in response")
        print(response)
    else:
        print(f"Ad {ad_id} not found in response")
        print(response)
else:
    print("No data were found in response")
    print(response)

Ad 1362986028521549 not found in response
{'data': []}


In [15]:
ad_id = "1362986028521549"
fields="id"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["BR"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_page_ids": ad_id,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    found_ad = False
    for ad in response["data"]:
        if ad["id"] == ad_id:
            found_ad = True
            break
    if found_ad:
        print(f"Found ad {ad_id} in response")
        print(response)
    else:
        print(f"Ad {ad_id} not found in response")
        print(response)
else:
    print("No data were found in response")
    print(response)

Ad 1362986028521549 not found in response
{'data': []}


**OC6: Can data from ads served by a specific advertiser be retrieved from the platform?**

*This item verifies whether it is possible to retrieve data from ads run by a specific advertiser, via their username or unique identifier. The assessment should review the ad repository documentation and test any available feature to retrieve data from an individual advertiser.*


Yes, by the advertiser page_id.

In [16]:
# Use this cell to develop an example where a request for ads from an advertiser is made.
# Please leave the result as the output of this cell.
advertiser = "651655564979252"
fields="id,page_id,page_name"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["BR"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_page_ids": advertiser,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    found_advertiser = False
    other_advertisers = 0
    for ad in response["data"]:
        if ad["page_id"] == advertiser:
            found_advertiser = True
        else:
            other_advertisers += 1
    if found_advertiser and other_advertisers == 0:
        print(f"Only ads from advertiser {advertiser} were found in response")
    elif found_advertiser and other_advertisers > 0:
        print(f"Found ads from advertiser {advertiser} and {other_advertisers} others in response")
    else:
        print(f"Advertiser {advertiser} not found in response")
else:
    print("No data were found in response")
    print(response)

Only ads from advertiser 651655564979252 were found in response


**OC7: Can ad data be retrieved from the platform using search terms?**

*This item verifies whether data on ad campaigns can be retrieved via individual or combined search terms, enabling the creation of a dataset composed of ads that mention those terms. The assessment should test search-related features to confirm that queries using keywords return relevant ad campaign data.*


Yes

In [14]:
# Use this cell to develop an example where a request for ads is made using a search term.
# Please leave the result as the output of this cell.
search_term = "saúde"
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["BR"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_terms": search_term,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response["data"])} ads in a single request using the search term {search_term}")
    df = pd.DataFrame(response["data"])
    display(df.head())
else:
    print("No ads were found")
    print(response)

Got 25 ads in a single request using the search term saúde


,id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,currency,...,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location,bylines,ad_creative_link_descriptions
0,679195485160563,2026-01-02,[📢 CONCURSO DA PREFEITURA DE JI-PARANÁ CONFIRM...,[WhatsApp],[api.whatsapp.com],2026-01-02,2026-01-03,"[{'country': 'BR', 'age_gender_breakdowns': [{...",1657,BRL,...,134017799788024,Professor Pinheiro Neto,"[facebook, instagram, audience_network, messen...","{'lower_bound': '0', 'upper_bound': '99'}","[18, 65]",All,"[{'name': 'Ji-Paraná, RO, Brasil', 'num_obfusc...","[{'key': 'BR', 'value': 1657}]",NaN,NaN
1,1493697551691716,2026-01-02,[📢 CONCURSO DA PREFEITURA DE JI-PARANÁ CONFIRM...,[WhatsApp],[api.whatsapp.com],2026-01-02,2026-01-03,"[{'country': 'BR', 'age_gender_breakdowns': [{...",4300,BRL,...,134017799788024,Professor Pinheiro Neto,"[facebook, instagram, audience_network, messen...","{'lower_bound': '0', 'upper_bound': '99'}","[18, 65]",All,"[{'name': 'Ji-Paraná, RO, Brasil', 'num_obfusc...","[{'key': 'BR', 'value': 4300}]",NaN,NaN
2,1382880096340198,2026-01-06,[📍 Um ano de mandato. Muitas histórias. Muita ...,NaN,NaN,2026-01-06,2026-01-07,"[{'country': 'BR', 'age_gender_breakdowns': [{...",4599,BRL,...,1817070968511062,Dudu Cunha,"[facebook, instagram]","{'lower_bound': '0', 'upper_bound': '99'}","[18, 65]",All,"[{'name': 'Indaial, SC, Brasil', 'num_obfuscat...","[{'key': 'BR', 'value': 4599}]",Dudu Cunha,NaN
3,4198014317154825,2026-01-07,[Hoje damos um passo muito importante para Bom...,[www.instagram.com],NaN,2026-01-07,NaN,"[{'country': 'BR', 'age_gender_breakdowns': [{...",1406,BRL,...,355964377812210,Eures Ribeiro,"[facebook, instagram]","{'lower_bound': '0', 'upper_bound': '99'}","[25, 60]",All,"[{'name': 'Bom Jesus da Lapa, BA, Brasil', 'nu...","[{'key': 'BR', 'value': 1406}]",Eures Ribeiro,NaN
4,3915139728786569,2026-01-07,[RETROSPECTIVA 2025 | Saúde de Maracaju\nJanei...,NaN,NaN,2026-01-07,NaN,"[{'country': 'BR', 'age_gender_breakdowns': [{...",404,BRL,...,261610264541037,Marcos Calderan,"[facebook, instagram]","{'lower_bound': '0', 'upper_bound': '99'}","[20, 65]",All,"[{'name': 'Maracaju (disambiguation), MS, Bras...","[{'key': 'BR', 'value': 404}]",Marcos Calderan,NaN


**OC8: Does the platform use locale-neutral data representations?**

*This item verifies whether locale-sensitive data (e.g., timestamps, currency, numbers) are provided in a locale-neutral format, or, if that is not possible, whether relevant locale metadata is included. The assessment should review the ad repository documentation and inspect sample responses to confirm the presence of standardized formats or accompanying metadata.*


Yes

In [15]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
query = "saúde"
fields="id,ad_creation_time,ad_delivery_start_time,ad_delivery_stop_time,currency"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["BR"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response["data"])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df.head())
else:
    print("No ads were found")
    print(response)


Got 25 ads in a single request


,id,ad_creation_time,ad_delivery_start_time,ad_delivery_stop_time,currency
0,679195485160563,2026-01-02,2026-01-02,2026-01-03,BRL
1,1493697551691716,2026-01-02,2026-01-02,2026-01-03,BRL
2,1382880096340198,2026-01-06,2026-01-06,2026-01-07,BRL
3,4198014317154825,2026-01-07,2026-01-07,NaN,BRL
4,3915139728786569,2026-01-07,2026-01-07,NaN,BRL


In [16]:
if "data" in response:
    df = pd.json_normalize(response["data"])
    df["ad_creation_time"] = pd.to_datetime( df["ad_creation_time"])
    df["ad_delivery_start_time"] = pd.to_datetime( df["ad_delivery_start_time"])
    df["ad_delivery_stop_time"] = pd.to_datetime( df["ad_delivery_stop_time"])
    schema = pa.DataFrameSchema(
        {
            "ad_creation_time": pa.Column(
                pa.DateTime, 
                nullable=True, 
                title="Ad creation time"
            ),
            "ad_delivery_start_time": pa.Column(
                pa.DateTime, 
                nullable=False, 
                title="Ad delivery start time"
            ),
            "ad_delivery_stop_time": pa.Column(
                pa.DateTime, 
                nullable=True, 
                title="Ad delivery stop time"
            ),
            "currency": pa.Column(
                pa.String,
                pa.Check.equal_to("BRL"), 
                nullable=False, 
                title="Currency"
            ),
        
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: Data is locale-neutral represented.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Data is not locale-neutral represented.")
        print(err)

✅ Validation Successful: Data is locale-neutral represented.


### COMPLETENESS

**OC9: Does the platform provide data that allows the identification of advertisers who ran ads?**

*This item verifies whether the platform discloses information on the advertisers responsible for the identified ads. The assessment should confirm whether the advertiser’s page name, URL, and unique identifier can be retrieved.*



Yes, page name and id can be found in response

In [17]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
query = "saúde"
fields="id,page_id,page_name"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["BR"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response["data"])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df.head())
else:
    print("No ads were found")
    print(response)


Got 25 ads in a single request


,id,page_id,page_name
0,679195485160563,134017799788024,Professor Pinheiro Neto
1,1493697551691716,134017799788024,Professor Pinheiro Neto
2,1382880096340198,1817070968511062,Dudu Cunha
3,880791421206058,715505761652752,Júnior Nogueira CG MS
4,1246509440716976,981639701954120,Marvel Maillet


In [18]:
if "data" in response:
    df = pd.DataFrame(response["data"])
    schema = pa.DataFrameSchema(
        {
            "page_id": pa.Column(
                pa.String, 
                nullable=False,  
                title="Page Id"
            ),
            "page_name": pa.Column(
                pa.String, 
                nullable=False,   
                title="Page Name"
            ),
        
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: Advertisers information columns are present and non-null.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Missing advertisers information.")
        print(err)

✅ Validation Successful: Advertisers information columns are present and non-null.


**OC10: Does the platform provide data on the funders who paid for ads?**

*This item verifies whether the platform provides data on who paid for ad campaigns. The assessment should confirm whether any sponsor information can be retrieved.*


Yes, the name of funder can be found in response.

In [19]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
query = "saúde"
fields="id,bylines"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["BR"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": False
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response["data"])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df.head())
else:
    print("No ads were found")
    print(response)

Got 25 ads in a single request


,id,bylines
0,679195485160563,NaN
1,1493697551691716,NaN
2,1382880096340198,Dudu Cunha
3,880791421206058,Júnior Nogueira CG MS
4,1246509440716976,Marvel Maillet


In [20]:
if "data" in response:
    df = pd.DataFrame(response["data"])
    schema = pa.DataFrameSchema(
        {
            "bylines": pa.Column(
                pa.String, 
                nullable=False,  
                title="Bylines"
            )
        
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: Funders information columns are present and non-null.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Missing funders information.")
        print(err)

❌ Validation Failed: Missing funders information.
{
    "SCHEMA": {
        "SERIES_CONTAINS_NULLS": [
            {
                "schema": null,
                "column": "bylines",
                "check": "not_nullable",
                "error": "non-nullable series 'bylines' contains null values:0    NaN1    NaNName: bylines, dtype: object"
            }
        ]
    }
}


**OC11: Does the platform provide data on the period during which ads were served?**

*This item verifies whether the platform provides data on the days on which ad campaigns ran. The assessment should review ad records to confirm that campaigns include start and end dates (or equivalent temporal markers) covering their active period*

Yes

In [21]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
query = "saúde"
fields="id,ad_delivery_start_time,ad_delivery_stop_time"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "INACTIVE",
            "ad_reached_countries": ["BR"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response["data"])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df.head())
else:
    print("No ads were found")
    print(response)

Got 25 ads in a single request


,id,ad_delivery_start_time,ad_delivery_stop_time
0,679195485160563,2026-01-02,2026-01-03
1,1493697551691716,2026-01-02,2026-01-03
2,1382880096340198,2026-01-06,2026-01-08
3,1591573105178400,2026-01-07,2026-01-07
4,1403648807811073,2026-01-07,2026-01-07


In [22]:
if "data" in response:
    df = pd.DataFrame(response["data"])
    df["ad_delivery_start_time"] = pd.to_datetime(df["ad_delivery_start_time"])
    df["ad_delivery_stop_time"] = pd.to_datetime(df["ad_delivery_stop_time"])
    schema = pa.DataFrameSchema(
        {
            "ad_delivery_start_time": pa.Column(
                pa.DateTime, 
                nullable=False,  
                title="Ad delivery start time"
            ),
            "ad_delivery_stop_time": pa.Column(
                pa.DateTime, 
                nullable=False,  
                title="Ad delivery stop time"
            )
        
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: Ads' run period dates are present and non-null.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Missing ads' run period dates.")
        print(err)


✅ Validation Successful: Ads' run period dates are present and non-null.


**OC12: Does the platform provide data on user engagement with ads?**

*This item verifies whether the platform provides data on the total number of user interactions with ad campaigns (e.g., likes, comments, shares, clicks). The assessment should review ad records to confirm that engagement metrics are available and clearly linked to each campaign.*


There's no field to retrieve user interactions. A complete list of available fields can be found in: https://developers.facebook.com/docs/graph-api/reference/archived-ad/

**OC13: Does the platform indicate whether ads were placed by verified or unverified advertisers?**

*This item verifies whether the platform clearly indicates whether advertisers were verified at the time their ads were served. The assessment should review ad records to confirm that a verification status field is present.*


There's no field that indicates if a advertiser is verified. A complete list of available fields can be found in: https://developers.facebook.com/docs/graph-api/reference/archived-ad/

### COMPLIANCE

**OC14: Does the platform flag ads that were removed due to violations of its guidelines or relevant legislation?**

*This item verifies whether the platform indicates when an ad has been moderated. At a minimum, the platform should provide the reason for removal and the date. The assessment should review ad records to confirm that moderated ads are flagged and that the corresponding moderation details are clearly documented.*



For this test we used the [Ad 31639644339013411](https://www.facebook.com/ads/library/?id=31639644339013411) that was previously removed from the platform.

In [23]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
query = "Instagram User 8714719066"
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["BR"],
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True # unmask set to TRUE
        }
response = request_to_api(session, payload, api_url)
df = pd.DataFrame(response["data"])
display(df)

,id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,currency,...,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location
0,31639644339013411,2025-10-12,"[Esse violador sexual chamado Hernane Alves, q...",[instagram.com],[instagram.com],2025-10-12,2025-10-12,"[{'country': 'BR', 'age_gender_breakdowns': [{...",469,BRL,...,"{'lower_bound': '0', 'upper_bound': '999'}",[pt],0,Instagram User 8714719066,[instagram],"{'lower_bound': '0', 'upper_bound': '99'}","[18, 65]",All,"[{'name': 'Brasil', 'num_obfuscated': 0, 'type...","[{'key': 'BR', 'value': 469}]"


In [24]:
query = "Instagram User 8714719066"
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["BR"],
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": False # unmask set to False
        }
response = request_to_api(session, payload, api_url)
df = pd.DataFrame(response["data"])
display(df)

,id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,...,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location
0,31639644339013411,2025-10-12,[This ad ran without a required disclaimer.],[This ad ran without a required disclaimer.],[This ad ran without a required disclaimer.],[This ad ran without a required disclaimer.],2025-10-12,2025-10-12,"[{'country': 'BR', 'age_gender_breakdowns': [{...",469,...,"{'lower_bound': '0', 'upper_bound': '999'}",[pt],0,Instagram User 8714719066,[instagram],"{'lower_bound': '0', 'upper_bound': '99'}","[18, 65]",All,"[{'name': 'Brasil', 'num_obfuscated': 0, 'type...","[{'key': 'BR', 'value': 469}]"


**OC15: Does the platform indicate whether ad creatives were generated using artificial intelligence?**

*This item verifies whether the platform flags ad campaigns in which AI played a role in producing the content. The assessment should review ad records to confirm the presence of a field or label indicating the use of AI in ad production.*
 


There's no field that indicates if an ad was created using AI. A complete list of available fields can be found in: https://developers.facebook.com/docs/graph-api/reference/archived-ad/

A test was made with [AD 351429144345136](
https://www.facebook.com/ads/library/?id=351429144345136) since a label for digitally created ads can be found in the Ad Library interface, but no label was found in the API response, as seen below.

In [8]:
page_id = "112494504974318"
ad_id = "351429144345136"
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["BR"],
            "ad_type": "ALL",
            "media_type": "ALL",
            "search_page_ids": page_id,
            "unmask_removed_content": True # unmask set to TRUE
        }
response = request_to_api(session, payload, api_url)
df = pd.DataFrame(response["data"])
display(df[df["id"]== ad_id])


,id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend
5,351429144345136,2024-01-23,[Assista o vídeo antes que seja tarde demais...],[pay.kiwify.com.br],NaN,NaN,2024-01-23,2024-01-23,Atualidades Brasileiras,BRL,NaN,NaN,{'lower_bound': '1000001'},"{'lower_bound': '0', 'upper_bound': '999'}",[pt],112494504974318,Atualidades Brasileiras,"[facebook, instagram]","{'lower_bound': '0', 'upper_bound': '99'}"


### CONSISTENCY

**OC23: Does the data retrieved by the API reflect what is displayed on the platform’s ad repository GUI?**

*This item verifies whether the data returned by the platform’s ad repository API corresponds to the information displayed on its GUI in all its levels of detail. It should be possible to identify in the API response information such as authorship, complete content, and other campaigning information (e.g., amount spent, impressions reached). The assessment should compare API responses with the GUI to confirm that key elements—such as authorship, full content, and campaign information (e.g., spending, impressions)—are consistently included.*


In [28]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
page_id = "145479165646274"
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["BR"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_page_ids": page_id,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
print(response["data"][0])

{'id': '1198664898977602', 'ad_creation_time': '2025-10-13', 'ad_creative_link_captions': ['casa.org.br', 'casa.org.br', 'casa.org.br', 'casa.org.br', 'casa.org.br', 'casa.org.br'], 'ad_creative_link_titles': ['Home - Fundo Casa Socioambiental', 'Home - Fundo Casa Socioambiental', 'Home - Fundo Casa Socioambiental', 'Home - Fundo Casa Socioambiental', 'Home - Fundo Casa Socioambiental', 'Home - Fundo Casa Socioambiental'], 'ad_delivery_start_time': '2025-10-13', 'ad_delivery_stop_time': '2025-10-13', 'age_country_gender_reach_breakdown': [{'country': 'BR', 'age_gender_breakdowns': [{'age_range': '18-24', 'male': 7, 'female': 17}, {'age_range': '25-34', 'male': 8, 'female': 24}, {'age_range': '35-44', 'male': 3, 'female': 2}, {'age_range': '45-54', 'male': 2, 'female': 1}, {'age_range': '55-64', 'male': 1}, {'age_range': '65+', 'male': 1}]}], 'br_total_reach': 66, 'currency': 'BRL', 'estimated_audience_size': {'lower_bound': '1000001'}, 'impressions': {'lower_bound': '0', 'upper_bound':

**OC24: Are the results returned by the platform consistently reproducible?**


*This item verifies whether data accessed and extracted via the platform’s ad repository at a given time is consistent with other collections performed similarly, including cases where content was deleted in the interim. The assessment should perform repeated queries to confirm reproducibility of results.*

Test instructions: Please, develop a test that collects ads for 5 times using the same request parameters and the same endpoint. Save each response in separate files.

In [29]:
# Use this cell, or more, to develop a process that collects ads more than one time using the same request parameters.
# Describe the endpoint and the search parameters used.
# Please add each response for each time a request is made in a file and save it in the data folder of this repository.
# The files naming pattern should be as stated below:
total_runs = 5 
for idx in range(total_runs):
    sleep(10)
    index = idx + 1
    filename = f"br-meta-ads-question-24-run-{index}-{datetime.now().isoformat()}"

    fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
    api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
    query = "saúde"
    payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["BR"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
    response = request_to_api(session, payload, api_url)

    with open(f"{FILEPATH}/{filename}.json", "w") as fout:
        json.dump(response, fout)

In [30]:
# Read files here to compare
files = sorted(glob.glob(f"{FILEPATH}/br-meta-ads-question-24*"))
all_files = []
for idx, file in enumerate(files):
    with open(file) as fin:
        data = json.load(fin)
    df = pd.DataFrame(data["data"])
    df["filename"] = file.split("/")[-1]
    all_files.append(df)
df_complete = pd.concat(all_files, ignore_index=True)
df_agg = df_complete.groupby("id").agg(total_occurance=("id", "count"),first_occurance_file=("filename", "min"), last_occurance_file=("filename", "max"),start_delivery_date_max=("ad_delivery_start_time", "max"))
display(df_agg)
    

,total_occurance,first_occurance_file,last_occurance_file,start_delivery_date_max
id,,,,
1091200799188675,5,br-meta-ads-question-24-run-1-2025-10-24T14:52...,br-meta-ads-question-24-run-5-2025-10-24T14:53...,2025-08-12
1125917189622189,5,br-meta-ads-question-24-run-1-2025-10-24T14:52...,br-meta-ads-question-24-run-5-2025-10-24T14:53...,2025-10-24
1143641761191508,5,br-meta-ads-question-24-run-1-2025-10-24T14:52...,br-meta-ads-question-24-run-5-2025-10-24T14:53...,2025-10-23
1143654121076613,5,br-meta-ads-question-24-run-1-2025-10-24T14:52...,br-meta-ads-question-24-run-5-2025-10-24T14:53...,2025-10-24
1308942483776309,5,br-meta-ads-question-24-run-1-2025-10-24T14:52...,br-meta-ads-question-24-run-5-2025-10-24T14:53...,2025-10-24
1327950462157977,5,br-meta-ads-question-24-run-1-2025-10-24T14:52...,br-meta-ads-question-24-run-5-2025-10-24T14:53...,2025-10-24
1328897998965479,5,br-meta-ads-question-24-run-1-2025-10-24T14:52...,br-meta-ads-question-24-run-5-2025-10-24T14:53...,2025-10-24
1333291514321785,5,br-meta-ads-question-24-run-1-2025-10-24T14:52...,br-meta-ads-question-24-run-5-2025-10-24T14:53...,2025-10-24
1529066891610595,5,br-meta-ads-question-24-run-1-2025-10-24T14:52...,br-meta-ads-question-24-run-5-2025-10-24T14:53...,2025-10-24


**OC25: Is the data returned by the platform consistent with the parameters and filters used in the request?**

*This item verifies whether the data retrieved through the ad repository accurately reflects the parameters and filters specified at the time of the request. The assessment should run test queries with different filters to confirm that results consistently match the requested conditions.*

Test instructions: Please, develop a process that request data using different parameter to the same endpoint. If possible, test at least 5 different parameters/filters. If the platform provides less than 5, use all available parameters/filters.

Save each response in separate files. 

In [31]:
# Use this cell, or more, to develop a process that collects ads more than one time using different parameters.
# Describe the endpoint and the search parameters used.
# Please add each response for each time a request is made in a file and save it in the data folder of this repository.
# The files naming pattern should be as stated below: 
# SET A LIST WITH A LEAST 5 PARAMETERS
parameters = [{"ad_delivery_date_min": "2025-10-01"}, 
                {"bylines": ["N3 News"]}, 
                {"delivery_by_region": ["Minas Gerais"]}, 
                {"estimated_audience_size_min":50000}, 
                {"languages": ["pt"]}]
for idx, param in enumerate(parameters):
    sleep(1)
    index = idx + 1 
    filename = f"br-meta-ads-question-25-run-{index}-{list(param.keys())[0]}-{datetime.now().isoformat()}"
    
    fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
    api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
    query = "saúde"
    payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["BR"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
    payload.update(param)
    response = request_to_api(session, payload, api_url)

    with open(f"{FILEPATH}/{filename}.json", "w") as fout:
        json.dump(response, fout)

In [29]:
# ad_delivery_date_min > 2025-10-01
with open(f"{FILEPATH}/br-meta-ads-question-25-run-1-ad_delivery_date_min-2025-10-24T14-53-42-960243.json") as fin:
    data = json.load(fin)
df = pd.DataFrame(data["data"])
df["ad_delivery_start_time"] = pd.to_datetime(df["ad_delivery_start_time"])
df["ad_delivery_stop_time"] = pd.to_datetime(df["ad_delivery_stop_time"])
schema = pa.DataFrameSchema(
    {
        "ad_delivery_stop_time": pa.Column(
            pa.DateTime, 
            checks=pa.Check.greater_than_or_equal_to("2025-10-01"),
            nullable=True,  
            title="Ad delivery end time"
        )
    },
    strict=False
)
try:
    schema.validate(df, lazy=True)
    print("✅ Validation Successful: All ads ran after the set ad_delivery_date_min parameter.")
except SchemaErrors as err:
    print("❌ Validation Failed: Found ads that ran earlier than the set ad_delivery_date_min parameter")
    print(err)




✅ Validation Successful: All ads ran after the set ad_delivery_date_min parameter.


In [33]:
# bylines: N3 News
with open(f"{FILEPATH}/br-meta-ads-question-25-run-2-bylines-2025-10-24T14:53:45.122750.json") as fin:
    data = json.load(fin)
if "data" in data:
    df = pd.DataFrame(data["data"])

    schema = pa.DataFrameSchema(
        {
            "bylines": pa.Column(
                checks=pa.Check.equal_to("N3 News"),
                nullable=False,  
                title="Bylines"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads have funders information as set in bylines parameter.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Found ads that are not from funders as set in bylines parameter")
        print(err)




✅ Validation Successful: All ads have funders information as set in bylines parameter.


In [34]:
# delivery_by_region: Minas Gerais
def extract_regions(x):
    if isinstance(x, list) and len(x) > 0: 
        return [d['region'] for d in x]
    return x

def check_delivery_region(regions):
    return regions.apply(lambda region_list: "Minas Gerais" in region_list)

with open(f"{FILEPATH}/br-meta-ads-question-25-run-3-delivery_by_region-2025-10-24T14:53:46.886280.json") as fin:
    data = json.load(fin)

if "data" in data:
    df = pd.DataFrame(data["data"])
    df["ad_delivery_regions"] = df["delivery_by_region"].apply(extract_regions)

    schema = pa.DataFrameSchema(
        {
            "ad_delivery_regions": pa.Column(
                checks=pa.Check(check_delivery_region),
                nullable=False,  
                title="Ad delivery regions"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads have region information as set in delivery_by_region parameter.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Found ads that don't have or are not from the regions as set in delivery_by_region parameter")
        print(err)





❌ Validation Failed: Found ads that don't have or are not from the regions as set in delivery_by_region parameter
{
    "SCHEMA": {
        "SERIES_CONTAINS_NULLS": [
            {
                "schema": null,
                "column": "ad_delivery_regions",
                "check": "not_nullable",
                "error": "non-nullable series 'ad_delivery_regions' contains null values:1    NaNName: ad_delivery_regions, dtype: object"
            }
        ]
    }
}


In [35]:
# estimated_audience_size_min > 50000
with open(f"{FILEPATH}/br-meta-ads-question-25-run-4-estimated_audience_size_min-2025-10-24T14:53:48.831131.json") as fin:
    data = json.load(fin)

if "data" in data:
    df = pd.json_normalize(data["data"])
    df["estimated_audience_size.lower_bound"] = df["estimated_audience_size.lower_bound"].astype(int)
   
    schema = pa.DataFrameSchema(
        {
            "estimated_audience_size.lower_bound": pa.Column(
                pa.Int, 
                checks=pa.Check.greater_than_or_equal_to(50000),
                nullable=False,  
                title="Estimated audience size - Lower"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads have audience size greater than the set estimated_audience_size_min parameter.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Found ads that have audience size lower than the set estimated_audience_size_min parameter")
        print(err)

✅ Validation Successful: All ads have audience size greater than the set estimated_audience_size_min parameter.


In [36]:
# language: pt
with open(f"{FILEPATH}/br-meta-ads-question-25-run-5-languages-2025-10-24T14:53:50.464787.json") as fin:
    data = json.load(fin)

if "data" in data:
    df = pd.DataFrame(data["data"])
    df["languages"] = df["languages"].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else x)


    schema = pa.DataFrameSchema(
        {
            "languages": pa.Column(
                checks=pa.Check.equal_to("pt"),
                nullable=False,  
                title="Languages"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads have information regarding language and it's the correct language set in language parameter.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Found ads that don't have or are not from the language set in the language parameter")
        print(err)





✅ Validation Successful: All ads have information regarding language and it's the correct language set in language parameter.


### RELEVANCE

**OC26: Does the platform allow the use of temporal filters to retrieve data on ads?**

*This item verifies whether the ad repository allows filtering ad campaign data based on the time period during which the ads were served. The assessment should test queries with temporal filters to confirm that results accurately reflect the specified date ranges.*



Yes

In [31]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,ad_creation_time,ad_delivery_start_time,ad_delivery_stop_time"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "saúde"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["BR"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True,
            "ad_delivery_date_max": "2025-10-15",
            "ad_delivery_date_min": "2025-10-01"
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response["data"])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df.head())
else:
    print("No ads were found")
    print(response)


Got 25 ads in a single request


,id,ad_creation_time,ad_delivery_start_time,ad_delivery_stop_time
0,1343436120628029,2025-10-13,2025-10-14,2025-10-31
1,1329186445408656,2025-10-15,2025-10-15,2025-10-28
2,1520121405790095,2025-10-15,2025-10-15,2025-11-18
3,1118237056708560,2025-10-15,2025-10-15,2025-10-17
4,1238582554760914,2025-10-15,2025-10-15,2025-10-21


In [32]:
df = pd.DataFrame(response["data"])
df["ad_delivery_start_time"] = pd.to_datetime(df["ad_delivery_start_time"])
start_date = datetime(year=2025, month=10, day=1)
end_date = datetime(year=2025, month=10, day=15)

schema = pa.DataFrameSchema({
    "ad_delivery_start_time": pa.Column(
        pa.DateTime,
        pa.Check(lambda s: s.between(start_date, end_date, inclusive="both"), 
                 name="DateRangeCheck"),
        nullable=False
    )
})
try:
    schema.validate(df, lazy=True)
    print("✅ Validation Successful: All ads are between date range.")

except SchemaErrors as err:
    print("❌ Validation Failed: Found ads outside date range.")
    print(err) 

✅ Validation Successful: All ads are between date range.


**OC27: Does the platform allow filtering advertising data by ad category?**

*This item verifies whether the ad repository allows filtering ad campaign data based on any possible categories assigned at the time of ad creation. The assessment should run test queries with category filters to confirm that results align with the selected classifications.*


In [39]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
# EMPLOYMENT_ADS
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "saúde"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["BR"],
            "ad_type": "EMPLOYMENT_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response["data"])} in EMPLOYMENT_ADS in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data found in EMPLOYMENT_ADS")
    print(response)

No data found in EMPLOYMENT_ADS
{'error': {'message': 'An unknown error has occurred.', 'type': 'OAuthException', 'code': 1, 'fbtrace_id': 'AN3ZaLbI-3JwW_FupCYkJMq'}}


In [33]:
# POLITICAL_AND_ISSUE_ADS
fields="id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_descriptions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,bylines,currency,delivery_by_region,demographic_distribution,estimated_audience_size,impressions,languages,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "saúde"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["BR"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response["data"])} in POLITICAL_AND_ISSUE_ADS in a single request")
    df = pd.DataFrame(response["data"])
    display(df.head())
else:
    print("No data found in POLITICAL_AND_ISSUE_ADS")
    print(response)

Got 25 in POLITICAL_AND_ISSUE_ADS in a single request


,id,ad_creation_time,ad_creative_bodies,ad_creative_link_captions,ad_creative_link_titles,ad_delivery_start_time,ad_delivery_stop_time,age_country_gender_reach_breakdown,br_total_reach,currency,...,page_id,page_name,publisher_platforms,spend,target_ages,target_gender,target_locations,total_reach_by_location,bylines,ad_creative_link_descriptions
0,679195485160563,2026-01-02,[📢 CONCURSO DA PREFEITURA DE JI-PARANÁ CONFIRM...,[WhatsApp],[api.whatsapp.com],2026-01-02,2026-01-03,"[{'country': 'BR', 'age_gender_breakdowns': [{...",1657,BRL,...,134017799788024,Professor Pinheiro Neto,"[facebook, instagram, audience_network, messen...","{'lower_bound': '0', 'upper_bound': '99'}","[18, 65]",All,"[{'name': 'Ji-Paraná, RO, Brasil', 'num_obfusc...","[{'key': 'BR', 'value': 1657}]",NaN,NaN
1,1493697551691716,2026-01-02,[📢 CONCURSO DA PREFEITURA DE JI-PARANÁ CONFIRM...,[WhatsApp],[api.whatsapp.com],2026-01-02,2026-01-03,"[{'country': 'BR', 'age_gender_breakdowns': [{...",4300,BRL,...,134017799788024,Professor Pinheiro Neto,"[facebook, instagram, audience_network, messen...","{'lower_bound': '0', 'upper_bound': '99'}","[18, 65]",All,"[{'name': 'Ji-Paraná, RO, Brasil', 'num_obfusc...","[{'key': 'BR', 'value': 4300}]",NaN,NaN
2,1382880096340198,2026-01-06,[📍 Um ano de mandato. Muitas histórias. Muita ...,NaN,NaN,2026-01-06,2026-01-08,"[{'country': 'BR', 'age_gender_breakdowns': [{...",6838,BRL,...,1817070968511062,Dudu Cunha,"[facebook, instagram]","{'lower_bound': '0', 'upper_bound': '99'}","[18, 65]",All,"[{'name': 'Indaial, SC, Brasil', 'num_obfuscat...","[{'key': 'BR', 'value': 6838}]",Dudu Cunha,NaN
3,880791421206058,2026-01-07,"[Sem apoio, sem mídia ao lado, sem político de...",[instagram.com],[Júnior Nogueira|Campo Grande MS 🇧🇷 (@067junio...,2026-01-08,NaN,"[{'country': 'BR', 'age_gender_breakdowns': [{...",2,BRL,...,715505761652752,Júnior Nogueira CG MS,[instagram],"{'lower_bound': '0', 'upper_bound': '99'}","[18, 65]",All,"[{'name': 'Campo Grande, MS, Brasil', 'num_obf...","[{'key': 'BR', 'value': 2}]",Júnior Nogueira CG MS,"[40K Followers, 3,675 Following, 1,184 Posts -..."
4,1246509440716976,2026-01-07,[Esporte vai muito além do físico! 💪🧠\n\nNo di...,[www.instagram.com],NaN,2026-01-08,NaN,"[{'country': 'BR', 'age_gender_breakdowns': [{...",114,BRL,...,981639701954120,Marvel Maillet,[instagram],"{'lower_bound': '0', 'upper_bound': '99'}","[18, 54]",All,"[{'name': 'Macaé, RJ, Brasil', 'num_obfuscated...","[{'key': 'BR', 'value': 114}]",Marvel Maillet,NaN


**OC28: Does the platform allow filtering advertising data by geographic location?**

*This item assesses whether the ad repository allows filtering data by one or more subnational geographic locations where the ads were served. The assessment should test queries with location filters to confirm that results match the specified areas.*

In [34]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,delivery_by_region"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "saúde"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["BR"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "delivery_by_region": ["Minas Gerais"],
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response["data"])} in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data found")
    print(response)

Got 25 in a single request


,id,delivery_by_region
0,910960558028015,"[{'percentage': '0.965649', 'region': 'Minas G..."
1,1166403722334381,"[{'percentage': '0.903846', 'region': 'Minas G..."
2,1370880787472275,"[{'percentage': '0.904615', 'region': 'Minas G..."
3,2694035557620618,"[{'percentage': '0.966891', 'region': 'Minas G..."
4,1574447840556095,"[{'percentage': '1', 'region': 'Minas Gerais'}]"
5,1425881249197949,"[{'percentage': '1', 'region': 'Minas Gerais'}]"
6,1371631417785083,"[{'percentage': '0.003273', 'region': 'Acre (s..."
7,1216217113941683,"[{'percentage': '1', 'region': 'Minas Gerais'}]"
8,26500408349560004,"[{'percentage': '1', 'region': 'Minas Gerais'}]"
9,1435337424859717,"[{'percentage': '1', 'region': 'Minas Gerais'}]"


In [35]:
# delivery_by_region: Minas Gerais
def extract_regions(x):
    if isinstance(x, list) and len(x) > 0:
        return [d['region'] for d in x]
    return x

def check_delivery_region(regions):
    return regions.apply(lambda region_list: "Minas Gerais" in region_list)

if "data" in response:
    df = pd.DataFrame(response["data"])
    df["ad_delivery_regions"] = df["delivery_by_region"].apply(extract_regions)

    schema = pa.DataFrameSchema(
        {
            "ad_delivery_regions": pa.Column(
                checks=pa.Check(check_delivery_region),
                nullable=False,  
                title="Ad delivery regions"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads came from the region set in the geographic location filter.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Found ads that don't have or are not from the region set in the geographic location filter.")
        print(err)





❌ Validation Failed: Found ads that don't have or are not from the region set in the geographic location filter.
{
    "SCHEMA": {
        "SERIES_CONTAINS_NULLS": [
            {
                "schema": null,
                "column": "ad_delivery_regions",
                "check": "not_nullable",
                "error": "non-nullable series 'ad_delivery_regions' contains null values:11    NaN21    NaN22    NaN23    NaNName: ad_delivery_regions, dtype: object"
            }
        ]
    }
}


### ACCURACY

**OC29: Does the platform provide age and gender data on the audiences of ads?**

*This item verifies whether the platform provides data on the age and gender of audiences reached. The assessment should review the ad records to confirm that these breakdowns are available and consistently reported*

In [36]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
# POLITICAL_AND_ISSUE_ADS
fields="id,age_country_gender_reach_breakdown,demographic_distribution"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "saúde"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["BR"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response["data"])} in POLITICAL_AND_ISSUE_ADS in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data found in POLITICAL_AND_ISSUE_ADS")
    print(response)

Got 25 in POLITICAL_AND_ISSUE_ADS in a single request


,id,age_country_gender_reach_breakdown,demographic_distribution
0,679195485160563,"[{'country': 'BR', 'age_gender_breakdowns': [{...","[{'percentage': '0.072393', 'age': '18-24', 'g..."
1,1493697551691716,"[{'country': 'BR', 'age_gender_breakdowns': [{...","[{'percentage': '0.065783', 'age': '18-24', 'g..."
2,1382880096340198,"[{'country': 'BR', 'age_gender_breakdowns': [{...","[{'percentage': '0.003588', 'age': '18-24', 'g..."
3,880791421206058,"[{'country': 'BR', 'age_gender_breakdowns': [{...",NaN
4,1246509440716976,"[{'country': 'BR', 'age_gender_breakdowns': [{...","[{'percentage': '0.096491', 'age': '18-24', 'g..."
5,25587382064204325,"[{'country': 'BR', 'age_gender_breakdowns': [{...",NaN
6,743972252096762,"[{'country': 'BR', 'age_gender_breakdowns': [{...",NaN
7,878475421436753,"[{'country': 'BR', 'age_gender_breakdowns': [{...",NaN
8,1060832366189130,"[{'country': 'BR', 'age_gender_breakdowns': [{...","[{'percentage': '0.069812', 'age': '18-24', 'g..."
9,729896076701596,"[{'country': 'BR', 'age_gender_breakdowns': [{...","[{'percentage': '0.010138', 'age': '18-24', 'g..."


In [37]:
if "data" in response:
    df = pd.json_normalize(response["data"])
    schema = pa.DataFrameSchema(
        {
            "age_country_gender_reach_breakdown": pa.Column( 
                nullable=False,  
                title="Age Country Gender Reach Breakdown"
            ),
            "demographic_distribution": pa.Column(
                nullable=False,  
                title="Target Ages"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads have information about age and gender about ad audience.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Missing ads information about age and gender.")
        print(err)


❌ Validation Failed: Missing ads information about age and gender.
{
    "SCHEMA": {
        "SERIES_CONTAINS_NULLS": [
            {
                "schema": null,
                "column": "demographic_distribution",
                "check": "not_nullable",
                "error": "non-nullable series 'demographic_distribution' contains null values:3     NaN5     NaN6     NaN7     NaN10    NaNName: demographic_distribution, dtype: object"
            }
        ]
    }
}


**OC30: Does the platform provide subnational geographic data on the audience reached by ads?**

*This item verifies whether the platform provides data on the subnational geographic location of audiences reached. The assessment should review the ad records to confirm that such location breakdowns are available and consistently reported.*

In [ ]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
# POLITICAL_AND_ISSUE_ADS
fields="id,delivery_by_region,total_reach_by_location"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "saúde"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["BR"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": True
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response["data"])} in POLITICAL_AND_ISSUE_ADS in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data found in POLITICAL_AND_ISSUE_ADS")
    print(response)

Got 25 in POLITICAL_AND_ISSUE_ADS in a single request


,id,target_locations,total_reach_by_location,delivery_by_region
0,1091200799188675,"[{'name': 'Brasil', 'num_obfuscated': 0, 'type...","[{'key': 'BR', 'value': 1}]",NaN
1,1143641761191508,"[{'name': 'Barretos, SP, Brasil', 'num_obfusca...","[{'key': 'BR', 'value': 10768}]","[{'percentage': '1', 'region': 'São Paulo (sta..."
2,825467789995623,"[{'name': 'Gravataí, RS, Brasil', 'num_obfusca...","[{'key': 'BR', 'value': 85051}]","[{'percentage': '1', 'region': 'Rio Grande do ..."
3,865738653285590,"[{'name': 'Jaboticabal, SP, Brasil', 'num_obfu...","[{'key': 'BR', 'value': 207}]","[{'percentage': '1', 'region': 'São Paulo (sta..."
4,1527753158572059,"[{'name': 'Taubaté, SP, Brasil', 'num_obfuscat...","[{'key': 'BR', 'value': 243}]","[{'percentage': '1', 'region': 'São Paulo (sta..."
5,1504316384051700,"[{'name': 'Taubaté, SP, Brasil', 'num_obfuscat...","[{'key': 'BR', 'value': 172}]","[{'percentage': '1', 'region': 'São Paulo (sta..."
6,1467539061198861,"[{'name': 'Juiz de Fora, MG, Brasil', 'num_obf...","[{'key': 'BR', 'value': 384}]","[{'percentage': '1', 'region': 'Minas Gerais'}]"
7,1362505475241922,"[{'name': 'Alfredo Chaves, ES, Brasil', 'num_o...","[{'key': 'BR', 'value': 2}]",NaN
8,1308942483776309,"[{'name': 'Riachão, Maranhao, Brazil, MA, Bras...","[{'key': 'BR', 'value': 479}]","[{'percentage': '0.968815', 'region': 'Maranhã..."
9,1572385253754603,"[{'name': 'São José do Rio Preto, SP, Brasil',...","[{'key': 'BR', 'value': 5}]",NaN


In [ ]:
def check_region_granularity(target_locations): 
    subnational_granularity = False
    if isinstance(target_locations, list) and len(target_locations) > 0:
        for location in target_locations:
            locations = location.get("key").split(",")
            if len(locations) > 1:
                subnational_granularity = True
                break
    return subnational_granularity

def validate_granularity_column(s):
    return s.apply(check_region_granularity)

if "data" in response:
    df = pd.json_normalize(response["data"])
    schema = pa.DataFrameSchema(
        {
            "delivery_by_region": pa.Column( 
                nullable=False,  
                title="Delivery by region"
            ),
            "total_reach_by_location": pa.Column(
                pa.Object,
                pa.Check(validate_granularity_column, error="Ad lacks required subnational granularity."),
                nullable=False,  
                title="Total reach by location"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads have subnational geographic information about ad audience.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Missing ads subnational geographic information.")
        print(err)


❌ Validation Failed: Missing ads subnational geographic information.
{
    "SCHEMA": {
        "SERIES_CONTAINS_NULLS": [
            {
                "schema": null,
                "column": "delivery_by_region",
                "check": "not_nullable",
                "error": "non-nullable series 'delivery_by_region' contains null values:0     NaN7     NaN9     NaN14    NaN19    NaNName: delivery_by_region, dtype: object"
            }
        ]
    },
    "DATA": {
        "DATAFRAME_CHECK": [
            {
                "schema": null,
                "column": "target_locations",
                "check": "Ad lacks required subnational granularity.",
                "error": "Column 'target_locations' failed element-wise validator number 0: Ad lacks required subnational granularity. failure cases: [{'name': 'Brasil', 'num_obfuscated': 0, 'type': 'countries', 'excluded': False}]"
            }
        ]
    }
}


**OC31: Does the platform include data on audience targeting criteria defined by advertisers?**

*This item verifies whether the platform provides data on audience targeting criteria defined by the advertiser when publishing ads (e.g., demographic and geographic segments, as well as interests, attitudes, behaviors, and keywords). The assessment should review ad records to confirm that these targeting parameters are available and consistently reported.*


There's no field that indicates the targeting defined by the advertiser. A complete list of available fields can be found in: https://developers.facebook.com/docs/graph-api/reference/archived-ad/

**OC32: Does the platform provide granular volume ranges for ad impressions?**

*This item verifies whether the ad repository provides impression values for ads, using ranges that closely approximate the actual numbers. Intervals should be no larger than 10% of the upper bound of the value range they represent. For example, values up to 1,000 impressions should be displayed in intervals no larger than 100; between 1,000 and 10,000 in intervals no larger than 1,000; between 10,000 and 100,000 in intervals no larger than 10,000; between 100,000 and 1 million or above, in intervals no larger than 100,000. The assessment should measure whether the reported intervals remain within this threshold across the different value ranges using the platform’s documentation or available data interfaces.*


In [47]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,impressions"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "saúde"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["BR"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": False
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response["data"])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data in response")
    print(response)


Got 25 ads in a single request


,id,impressions
0,1091200799188675,"{'lower_bound': '0', 'upper_bound': '999'}"
1,1143641761191508,"{'lower_bound': '10000', 'upper_bound': '14999'}"
2,825467789995623,"{'lower_bound': '80000', 'upper_bound': '89999'}"
3,1308942483776309,"{'lower_bound': '0', 'upper_bound': '999'}"
4,1529066891610595,"{'lower_bound': '2000', 'upper_bound': '2999'}"
5,2645039575849197,"{'lower_bound': '0', 'upper_bound': '999'}"
6,803753479032404,"{'lower_bound': '1000', 'upper_bound': '1999'}"
7,1333291514321785,"{'lower_bound': '0', 'upper_bound': '999'}"
8,1742057633153172,"{'lower_bound': '0', 'upper_bound': '999'}"
9,666425706548110,"{'lower_bound': '0', 'upper_bound': '999'}"


In [48]:
if response["data"]:
    df = pd.json_normalize(response["data"])
    df["impressions.upper_bound"]= df["impressions.upper_bound"].astype(int)
    df["percentage"] = round((df["impressions.upper_bound"].astype(int) - df["impressions.lower_bound"].astype(int) ) / df["impressions.upper_bound"].astype(int)  * 100, 1)
    schema = pa.DataFrameSchema(
        {
            "percentage": pa.Column( 
                pa.Float,
                pa.Check.less_than_or_equal_to(10),
                nullable=False, 
                title="Interval between impressions values percentage"
            ),
            "impressions.upper_bound": pa.Column(
                nullable=False,  
                title="Impressions - Upper"
            ),
            "impressions.lower_bound": pa.Column(
                nullable=False,  
                title="Impressions - Lower"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads have impressions granularity in the expected range.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Impressions granularity between lower and upper bound are larger than 10% of the upper bound")
        print(err)


❌ Validation Failed: Impressions granularity between lower and upper bound are larger than 10% of the upper bound
{
    "DATA": {
        "DATAFRAME_CHECK": [
            {
                "schema": null,
                "column": "percentage",
                "check": "less_than_or_equal_to(10)",
                "error": "Column 'percentage' failed element-wise validator number 0: less_than_or_equal_to(10) failure cases: 100.0, 33.3, 11.1, 100.0, 33.3, 100.0, 50.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 50.0, 100.0"
            }
        ]
    }
}


**OC33: Does the platform provide granular investment ranges for ad spending?**

*This item verifies whether the ad repository provides spending values for ads, using ranges that closely approximate the actual amounts. Intervals should be no larger than 10% of the upper bound of the value range they represent. For example, values up to $100 should be displayed in intervals no larger than $10; between $100 and $1,000 in intervals no larger than $100; and between $1,000 and $10,000 in intervals no larger than $1,000. The assessment should measure whether the reported intervals remain within this threshold across the different value ranges using the platform’s documentation or available data interfaces.*


In [49]:
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
fields="id,spend"
api_url=f"https://graph.facebook.com/v23.0/ads_archive?fields={fields}"
query = "saúde"
payload={
            "access_token": ACCESS_TOKEN,
            "ad_active_status": "ALL",
            "ad_reached_countries": ["BR"],
            "ad_type": "POLITICAL_AND_ISSUE_ADS",
            "media_type": "ALL",
            "search_terms": query,
            "unmask_removed_content": False
        }
response = request_to_api(session, payload, api_url)
if "data" in response:
    print(f"Got {len(response["data"])} ads in a single request")
    df = pd.DataFrame(response["data"])
    display(df)
else:
    print("No data in response")
    print(response)


Got 25 ads in a single request


,id,spend
0,1091200799188675,"{'lower_bound': '0', 'upper_bound': '99'}"
1,1143641761191508,"{'lower_bound': '0', 'upper_bound': '99'}"
2,825467789995623,"{'lower_bound': '100', 'upper_bound': '199'}"
3,1308942483776309,"{'lower_bound': '0', 'upper_bound': '99'}"
4,1529066891610595,"{'lower_bound': '0', 'upper_bound': '99'}"
5,2645039575849197,"{'lower_bound': '0', 'upper_bound': '99'}"
6,803753479032404,"{'lower_bound': '0', 'upper_bound': '99'}"
7,1333291514321785,"{'lower_bound': '0', 'upper_bound': '99'}"
8,1742057633153172,"{'lower_bound': '0', 'upper_bound': '99'}"
9,666425706548110,"{'lower_bound': '0', 'upper_bound': '99'}"


In [50]:
if response["data"]:
    df = pd.json_normalize(response["data"])
    df["spend.upper_bound"]= df["spend.upper_bound"].astype(int)
    df["spend.lower_bound"]= df["spend.lower_bound"].astype(int)
    df["percentage"] = round((df["spend.upper_bound"] - df["spend.lower_bound"] ) / df["spend.upper_bound"]  * 100, 1)
    schema = pa.DataFrameSchema(
        {
            "percentage": pa.Column( 
                pa.Float,
                pa.Check.less_than_or_equal_to(10),
                nullable=False, 
                title="Interval between spend values percentage"
            ),
            "spend.upper_bound": pa.Column(
                nullable=False,  
                title="Spend - Upper"
            ),
            "spend.lower_bound": pa.Column(
                nullable=False,  
                title="Spend - Lower"
            )
        },
        strict=False
    )
    try:
        schema.validate(df, lazy=True)
        print("✅ Validation Successful: All ads have spend granularity in the expected range.")
    except SchemaErrors as err:
        print("❌ Validation Failed: Spend granularity between lower and upper bound are larger than 10% of the upper bound")
        print(err)


❌ Validation Failed: Spend granularity between lower and upper bound are larger than 10% of the upper bound
{
    "DATA": {
        "DATAFRAME_CHECK": [
            {
                "schema": null,
                "column": "percentage",
                "check": "less_than_or_equal_to(10)",
                "error": "Column 'percentage' failed element-wise validator number 0: less_than_or_equal_to(10) failure cases: 100.0, 100.0, 49.7, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0"
            }
        ]
    }
}
